# bordeus — valutazione RAG e confronto system prompt

Notebook per valutare la qualità del retrieval e confrontare varianti di
system prompt sullo stesso vector store che usa il bot in produzione —
senza dover passare da Telegram per ogni iterazione.

Riusa il codice reale del bot (`bordeus_bot.rag`: `QUERY_INSTRUCTION`,
`vocabolario_filter`, `calendario_filter`, `SYSTEM_PROMPT_TEMPLATE`),
non lo duplica: se qui un prompt funziona meglio, il passo successivo è
copiarlo in `bot/src/bordeus_bot/rag.py`.

**Retrieval a due fasi**: il bot recupera prima il contesto sul
*vocabolario* (dove va smaltito l'oggetto) usando la domanda originale,
poi usa quel contesto come query per recuperare il *calendario* di
raccolta pertinente (il giorno di passaggio) — così la risposta include
sempre anche il giorno quando disponibile, non solo su richiesta
esplicita. Vedi il docstring di `bot/rag.py` per il perché di due query
invece di una sola.

**Prerequisito**: un'area già popolata dalla pipeline di ingestion
(vedi `../../ingestion/notebooks/ingest.ipynb` o
`uv run bordeus-ingest ...`) — questo notebook non scrive nulla, solo
legge e interroga.

0. **Estrazione** — dalla domanda naturale dell'utente, la sola
   descrizione dell'oggetto (stesso passaggio del bot vero, non
   saltato: la query di retrieval non è mai l'intera domanda)
1. **Retrieval** — per un set di domande di prova, mostra i chunk
   recuperati in entrambe le fasi e la loro distanza (più bassa = più
   simile)
2. **Confronto system prompt** — stesse domande, stesso contesto
   recuperato, risposte fianco a fianco per ciascuna variante di prompt
3. **Controlli automatici di base** — euristiche semplici (risposta non
   vuota, non un semplice eco della domanda, niente artefatti di
   template, lingua plausibile) per un primo filtro prima della lettura
   manuale

In [ ]:
import logging
import os
import sys

sys.path.insert(0, "../../common/src")
sys.path.insert(0, "../src")

from dotenv import load_dotenv

load_dotenv("../.env")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

import pandas as pd
from bordeus_bot import i18n, identify
from bordeus_bot.rag import (
    QUERY_INSTRUCTION,
    SYSTEM_PROMPT_TEMPLATE,
    build_question,
    calendario_filter,
    vocabolario_filter,
)
from bordeus_common.embed import get_embeddings
from bordeus_common.vectorstore import get_vectorstore
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

pd.set_option("display.max_colwidth", None)

## Configurazione

`area_id` deve corrispondere a un'area già ingerita. `comune_id`
è opzionale: lascialo vuoto (`""`) per vedere solo il contenuto
condiviso dell'area, oppure imposta un comune reale per includere anche
il suo contenuto specifico (es. un calendario) — vedi
`comune_filter` in `bot/rag.py`.

In [ ]:
database_url = os.environ["DATABASE_URL"]

area_id = "sub-ato-e"      # cambia con un'area realmente ingerita
comune_id = "donnas"              # "" = solo contenuto condiviso; oppure uno id di comune reale dell'area

ollama_base_url = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
ollama_model = os.environ.get("OLLAMA_MODEL", "gemma3:4b")

# Lingua in cui il bot deve rispondere (uno tra "it", "fr", "en", "es",
# "de" — vedi bordeus_bot.i18n.SUPPORTED_LANGUAGES). Il vero bot la
# ricava da update.effective_user.language_code; qui la impostiamo a
# mano per poter confrontare le risposte nella stessa lingua su tutte
# le domande.
test_language = "it"

embeddings = get_embeddings(query_instruction=QUERY_INSTRUCTION)
vectorstore = get_vectorstore(database_url, area_id, embeddings)
llm = ChatOllama(model=ollama_model, base_url=ollama_base_url, temperature=0.2)

comune_display = repr(comune_id) if comune_id else "(nessuno, solo contenuto condiviso)"
print(f"Area: {area_id!r}  |  Comune: {comune_display}  |  Modello: {ollama_model!r}")

## Domande di prova

Placeholder generiche — sostituiscile con domande pertinenti ai
contenuti realmente ingeriti per la tua area (nomi di materiali/oggetti
che sai essere trattati nelle guide, più qualche domanda "trabocchetto"
fuori tema per verificare che il bot ammetta di non saperlo invece di
inventare). `comune_id` per riga è opzionale: se assente/`None`, usa
quello configurato sopra — utile per domande specifiche di un comune
diverso da quello di default (es. un calendario).

In [ ]:
eval_questions = [
    {"domanda": "Come smaltisco una bottiglia di plastica?"},
    {"domanda": "Dove butto le pile esaurite?"},
    {"domanda": "Come devo smaltire dei vecchi vestiti?"},
    {"domanda": "Dove butto una tazza in ceramica rotta?"},
    {"domanda": "Dove butto un set di piatti in ceramica?"},
    {"domanda": "Come smaltisco farmaci scaduti?"},
    # Oggetto tipicamente NON raccolto porta a porta (es. va
    # all'ecocentro): la fase 2 (calendario) non dovrebbe trovare nulla
    # di pertinente — un buon prompt deve ometterlo, non inventarlo.
    {"domanda": "Posso conferire un elettrodomestico rotto nel cassonetto normale?"},
    # Domanda "trabocchetto", fuori tema: un buon system prompt deve
    # ammettere di non saperlo, non inventare una risposta plausibile.
    {"domanda": "Qual è la ricetta della polenta valdostana?"},
]

for q in eval_questions:
    q.setdefault("comune_id", comune_id)

print(f"{len(eval_questions)} domande di prova")

## Step 0 — Estrazione della descrizione dell'oggetto

Replica il primo passo del bot vero (`identify.identify_object_from_text`),
non salta direttamente al retrieval: in produzione la query sul vector
store non è mai la domanda intera dell'utente, è solo la descrizione
dell'oggetto estratta da essa — usare la frase intera introdurrebbe
rumore (saluti, formule di cortesia, la struttura della domanda) nella
ricerca per similarità. Una domanda "trabocchetto" fuori tema (nessun
oggetto menzionato) deve restituire `None`: quella domanda salta
retrieval e generazione, esattamente come nel bot vero — non è un
errore, è il comportamento corretto.

In [ ]:
for q in eval_questions:
    descrizione = identify.identify_object_from_text(llm, q["domanda"])
    q["descrizione_oggetto"] = descrizione
    esito = repr(descrizione) if descrizione is not None else "None (nessun oggetto riconosciuto)"
    print(f"{q['domanda']!r}\n  -> {esito}\n")

## Step 1 — Qualità del retrieval (due fasi)

Replica esattamente il retrieval a due passaggi usato in produzione
(`bot/rag.py`, `answer_question`), non una versione semplificata —
compreso l'uso della sola **descrizione dell'oggetto** (estratta allo
Step 0) come query, non la domanda intera:

1. **Vocabolario** (`vocabolario_filter`: tutto tranne `tipo="calendario"`)
   — la descrizione dell'oggetto cerca qui, non la domanda originale.
2. **Calendario** (`calendario_filter`: solo `tipo="calendario"`, più il
   filtro per comune) — usa come query il TESTO recuperato al passaggio
   1, non la descrizione dell'oggetto (il vocabolario di solito nomina
   il materiale/bidone, un match migliore contro il calendario).

`similarity_search_with_score` (distanza coseno di default in
`PGVector` — verificato leggendo il sorgente: la strategia non è
impostata esplicitamente in `get_vectorstore`, quindi vale il default
della libreria): **più basso = più simile**. Se già la fase 1 recupera
chunk scadenti, nessun prompt potrà rimediarci a valle — e se la fase 2
non trova nulla di pertinente (es. l'oggetto va all'ecocentro, non è
raccolto porta a porta), è normale e atteso, non un errore.

Le domande senza un oggetto riconosciuto allo Step 0 (`descrizione_oggetto
is None`) vengono saltate: nel bot vero non arrivano nemmeno al
retrieval, si ferma prima con un messaggio che chiede di riformulare.

In [ ]:
def show_retrieval(descrizione_oggetto: str, comune_id_domanda: str, k_vocabolario: int = 6, k_calendario: int = 4) -> dict:
    print(f"Descrizione oggetto: {descrizione_oggetto!r}")
    print(f"Filtrato per comune: {comune_id_domanda!r}\n")

    vocabolario_risultati = vectorstore.similarity_search_with_score(
        descrizione_oggetto, k=k_vocabolario, filter=vocabolario_filter(comune_id_domanda)
    )
    print("--- Fase 1: vocabolario (query = descrizione oggetto) ---")
    for doc, distanza in vocabolario_risultati:
        print(f"  distanza={distanza:.4f}  [{doc.metadata.get('tipo')}]")
        print(f"    {doc.page_content[:180]!r}")
        print(f"    fonte: {doc.metadata.get('source_url')}\n")

    vocabolario_context = "\n\n".join(doc.page_content for doc, _ in vocabolario_risultati)

    calendario_risultati = []
    if vocabolario_context:
        calendario_risultati = vectorstore.similarity_search_with_score(
            vocabolario_context, k=k_calendario, filter=calendario_filter(comune_id_domanda)
        )
    print("--- Fase 2: calendario (query = testo recuperato in fase 1) ---")
    if not calendario_risultati:
        print("  (nessun risultato — normale se l'oggetto non è raccolto porta a porta)\n")
    for doc, distanza in calendario_risultati:
        tag_comune = doc.metadata.get("comune_id") or "area condivisa"
        print(f"  distanza={distanza:.4f}  [{tag_comune}]")
        print(f"    {doc.page_content[:180]!r}")
        print(f"    fonte: {doc.metadata.get('source_url')}\n")

    return {"vocabolario": vocabolario_risultati, "calendario": calendario_risultati}


retrieval_cache = {}
for q in eval_questions:
    if q["descrizione_oggetto"] is None:
        print(f"'{q['domanda']}' — nessun oggetto riconosciuto, salto il retrieval (comportamento atteso)")
        print("-" * 80)
        continue
    chiave = (q["domanda"], q["comune_id"])
    retrieval_cache[chiave] = show_retrieval(q["descrizione_oggetto"], q["comune_id"])
    print("-" * 80)

## Step 2 — Varianti di system prompt

`"attuale"` è importato direttamente da `bot/rag.py`
(`SYSTEM_PROMPT_TEMPLATE`) — è il vero prompt in produzione, non una
copia che potrebbe disallinearsi. Le altre sono punti di partenza da
modificare/estendere liberamente: l'idea è generarne quante ne servono,
confrontarle sulle stesse domande, e copiare la vincitrice in
`bot/rag.py` una volta scelta.

In [ ]:
PROMPT_VARIANTS = {
    "attuale": SYSTEM_PROMPT_TEMPLATE,

    "piu_conciso": """\
Sei un assistente che rappresenta un'azienda di gestione rifiuti.
Rispondi ESCLUSIVAMENTE in base al contesto sottostante, non usare conoscenza generale sullo smaltimento rifiuti anche se pensi di saperla: le regole cambiano da comune a comune.
Rispondi SEMPRE in {lingua}, indipendentemente dalla lingua del contesto sottostante.
Rispondi in UNA frase, massimo due. Niente preamboli, niente ripetizioni della domanda.
Usa "Modalità di smaltimento" per il bidone/modalità giusta. Se "Calendario di raccolta" contiene un giorno pertinente, aggiungilo in coda, altrimenti ometti.
Se non c'è abbastanza informazione, dillo in poche parole.

Modalità di smaltimento:
{vocabolario_context}

Calendario di raccolta:
{calendario_context}
""",

    "few_shot": """\
Sei un assistente cordiale che rappresenta un'azienda di gestione rifiuti. Rispondi ESCLUSIVAMENTE in base al contesto sottostante, non usare conoscenza generale sullo smaltimento rifiuti anche se pensi di saperla: le regole cambiano da comune a comune. Rispondi SEMPRE in {lingua}, come negli esempi (gli esempi sono in italiano solo per mostrare lo stile, non la lingua da usare), includendo sempre il giorno di raccolta se disponibile nel calendario.

Esempio 1
Domanda: Dove butto una bottiglia di vetro?
Risposta: Nel bidone del vetro, senza tappo. A Donnas il vetro passa il martedì.

Esempio 2
Domanda: Qual è la capitale della Francia?
Risposta: Non è una domanda sui rifiuti, non posso aiutarti con questo.

Ora rispondi tu, usando "Modalità di smaltimento" e, quando pertinente, "Calendario di raccolta". Se non conosci la risposta, dillo esplicitamente.

Modalità di smaltimento:
{vocabolario_context}

Calendario di raccolta:
{calendario_context}
""",
}

print("Varianti definite:", list(PROMPT_VARIANTS.keys()))
print(f"Lingua di risposta per il confronto: {test_language!r} ({i18n.language_name(test_language)})")

## Step 3 — Esegui il confronto

Stesso contesto recuperato (vocabolario + calendario) per tutte le
varianti sulla stessa domanda — riusa `retrieval_cache` dello Step 1,
non ri-fa la ricerca — così le differenze nelle risposte vengono solo
dal prompt, non da un retrieval diverso per caso. Con
`len(eval_questions) * len(PROMPT_VARIANTS)` chiamate all'LLM, può
richiedere qualche minuto a seconda del modello.

In [ ]:
def answer_with_variant(prompt_template: str, descrizione_oggetto: str, comune_id_domanda: str) -> str:
    cache = retrieval_cache[(descrizione_oggetto_to_domanda[descrizione_oggetto], comune_id_domanda)]
    vocabolario_context = "\n\n".join(doc.page_content for doc, _ in cache["vocabolario"])
    calendario_context = "\n\n".join(doc.page_content for doc, _ in cache["calendario"])
    system_prompt = prompt_template.format(
        lingua=i18n.language_name(test_language),
        vocabolario_context=vocabolario_context or "Nessuna informazione trovata.",
        calendario_context=calendario_context or "Nessuna informazione di calendario trovata.",
    )
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=[{"type": "text", "text": build_question(descrizione_oggetto)}]),
    ]
    return llm.invoke(messages).content


# Serve per risalire dalla descrizione (chiave usata da answer_with_variant,
# coerente con build_question) alla domanda originale (chiave di
# retrieval_cache) — un piccolo indice di comodo, non serve altrove.
descrizione_oggetto_to_domanda = {
    q["descrizione_oggetto"]: q["domanda"] for q in eval_questions if q["descrizione_oggetto"] is not None
}

risultati_confronto = []
for q in eval_questions:
    if q["descrizione_oggetto"] is None:
        print(f"'{q['domanda']}' — nessun oggetto riconosciuto allo Step 0, salto il confronto prompt (comportamento atteso: il bot vero si ferma qui)")
        continue
    riga = {"domanda": q["domanda"]}
    for nome_variante, template in PROMPT_VARIANTS.items():
        riga[nome_variante] = answer_with_variant(template, q["descrizione_oggetto"], q["comune_id"])
    risultati_confronto.append(riga)

df_confronto = pd.DataFrame(risultati_confronto).set_index("domanda")
print(f"\n{len(risultati_confronto)} domande x {len(PROMPT_VARIANTS)} varianti generate")

### Confronto leggibile, una domanda alla volta

Una tabella larga con colonne di testo lungo è scomoda da leggere in un
notebook — stampiamo invece un blocco per domanda, con tutte le
varianti sotto, una sopra l'altra.

In [ ]:
for domanda, riga in df_confronto.iterrows():
    print("=" * 80)
    print(f"DOMANDA: {domanda}\n")
    for nome_variante in PROMPT_VARIANTS:
        print(f"--- {nome_variante} ---")
        print(riga[nome_variante])
        print()

## Step 4 — Controlli automatici di base

Euristiche semplici, non un giudizio di qualità vero e proprio: servono
solo a individuare rapidamente le risposte chiaramente problematiche
(vuote, che si limitano a ripetere la domanda, con artefatti di
template non sostituiti, o in una lingua diversa dall'italiano) prima
di leggere tutto a mano. Un controllo passato non garantisce che la
risposta sia *corretta* — solo che non ha questi problemi specifici.

In [ ]:
_PAROLE_ITALIANE_COMUNI = {"il", "la", "di", "che", "per", "non", "un", "una", "sono", "puoi", "è", "e"}


def controlli_base(risposta: str, domanda: str) -> dict:
    parole = set(risposta.lower().replace(",", " ").replace(".", " ").split())
    return {
        "non_vuota": len(risposta.strip()) > 0,
        "non_eco_domanda": risposta.strip().lower() != domanda.strip().lower(),
        "niente_placeholder": "{context}" not in risposta,
        "lunghezza_ragionevole": 5 <= len(risposta.split()) <= 200,
        "sembra_italiano": len(parole & _PAROLE_ITALIANE_COMUNI) > 0,
    }


righe_scorecard = []
for domanda, riga in df_confronto.iterrows():
    for nome_variante in PROMPT_VARIANTS:
        esiti = controlli_base(riga[nome_variante], domanda)
        righe_scorecard.append({
            "domanda": domanda,
            "variante": nome_variante,
            **esiti,
            "tutti_ok": all(esiti.values()),
        })

df_scorecard = pd.DataFrame(righe_scorecard)

print("Riepilogo per variante (quota di controlli superati su tutte le domande):\n")
print(df_scorecard.groupby("variante")["tutti_ok"].mean().sort_values(ascending=False))

print("\nRighe con almeno un controllo fallito (da leggere con attenzione):")
df_scorecard[~df_scorecard["tutti_ok"]]

## Estendere la valutazione

- **Più domande**: aggiungi voci a `eval_questions` — includi sempre
  qualche domanda fuori tema, è il modo più semplice per verificare che
  il prompt non "inventi" risposte.
- **Più varianti**: aggiungi voci a `PROMPT_VARIANTS` — qualunque
  stringa con un placeholder `{context}` funziona.
- **Un vero ground truth**: se hai risposte di riferimento scritte a
  mano per alcune domande, un prossimo passo naturale è confrontarle
  con quelle generate (anche solo a occhio, o con un LLM-as-judge — un
  secondo prompt che chiede a un modello di valutare la risposta contro
  il riferimento su una scala 1-5). Non implementato qui: richiede cura
  nel prompt di valutazione per essere affidabile, e un ground truth che
  al momento questo notebook non presuppone.